In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [4]:
import joblib
import os

In [5]:
# ============================================================
# PROJECT PATHS
# ============================================================

project_root = Path.cwd().parent
model_dir=project_root/"models"
recommendation_dir = (
    project_root
    / "data"
    / "processed"
    / "recommendation"
)


recommendation_dir = (
    project_root
    / "data"
    / "processed"
    / "recommendation"
)



In [6]:
import pandas as pd

products = pd.read_csv(
    recommendation_dir/"products_processed_popularity.csv",
    low_memory=False
)

reviews = pd.read_csv(
    recommendation_dir/"reviews_processed_popularity.csv",
    low_memory=False
)

In [7]:
tfidf = joblib.load(
    model_dir/"recommendation/content_based/tfidf_vectorizer.pkl"
)


content_similarity = joblib.load(
    model_dir/"recommendation/content_based/cosine_similarity.pkl"
)


content_products = joblib.load(
    model_dir/"recommendation/content_based/product_indices.pkl"
)

In [8]:
type(tfidf), type(content_similarity), type(content_products)

(sklearn.feature_extraction.text.TfidfVectorizer,
 numpy.ndarray,
 pandas.core.frame.DataFrame)

In [9]:
content_products = content_products.reset_index()

In [10]:
content_products.head()

,index,product_idx,product_id,product_name,brand_name,primary_category,price_usd
0,0,0,P473671,Fragrance Discovery Set,19-69,Fragrance,35.0
1,1,1,P473668,La Habana Eau de Parfum,19-69,Fragrance,195.0
2,2,2,P473662,Rainbow Bar Eau de Parfum,19-69,Fragrance,195.0
3,3,3,P473660,Kasbah Eau de Parfum,19-69,Fragrance,195.0
4,4,4,P473658,Purple Haze Eau de Parfum,19-69,Fragrance,195.0


In [11]:
content_products.columns = [
    "product_idx",
    "product_id"
]

ValueError: Length mismatch: Expected axis has 7 elements, new values have 2 elements

In [ ]:
content_products.dtypes

product_id    int64
dtype: object

In [ ]:
content_products.head()

,product_idx,product_id
0,Fragrance Discovery Set,0
1,La Habana Eau de Parfum,1
2,Rainbow Bar Eau de Parfum,2
3,Kasbah Eau de Parfum,3
4,Purple Haze Eau de Parfum,4


In [ ]:
content_products = products[
    [
        "product_id",
        "product_name",
        "brand_name",
        "primary_category",
        "price_usd"
    ]
].copy()

In [ ]:
content_products["product_idx"] = range(len(content_products))

In [ ]:
content_products.head()

,product_id,product_name,brand_name,primary_category,price_usd,product_idx
0,P473671,Fragrance Discovery Set,19-69,Fragrance,35.0,0
1,P473668,La Habana Eau de Parfum,19-69,Fragrance,195.0,1
2,P473662,Rainbow Bar Eau de Parfum,19-69,Fragrance,195.0,2
3,P473660,Kasbah Eau de Parfum,19-69,Fragrance,195.0,3
4,P473658,Purple Haze Eau de Parfum,19-69,Fragrance,195.0,4


In [ ]:
content_products = content_products[
    [
        "product_idx",
        "product_id",
        "product_name",
        "brand_name",
        "primary_category",
        "price_usd"
    ]
]

In [ ]:
content_products.iloc[0]

product_idx                               0
product_id                          P473671
product_name        Fragrance Discovery Set
brand_name                            19-69
primary_category                  Fragrance
price_usd                              35.0
Name: 0, dtype: object

In [ ]:
products.iloc[0][
    [
        "product_id",
        "product_name"
    ]
]

product_id                      P473671
product_name    Fragrance Discovery Set
Name: 0, dtype: object

In [ ]:
joblib.dump(
    content_products,
    model_dir/"recommendation/content_based/product_indices.pkl"
)

['c:\\Users\\ujjwa\\OneDrive\\Documents\\beauty-recommender\\models\\recommendation\\content_based\\product_indices.pkl']

In [ ]:
item_knn = joblib.load(
    model_dir/"recommendation/collab/item_knn.pkl"
)


item_user_matrix = joblib.load(
   model_dir/"recommendation/collab/item_user_matrix.pkl"
)


cf_products = pd.read_csv(
    model_dir/"recommendation/collab/product_map.csv"
)


cf_products.head()

,product_idx,product_id,product_name,brand_name,primary_category,secondary_category
0,199,P394639,The True Cream Aqua Bomb,belif,Skincare,Moisturizers
1,236,P4016,Acne Control Clarifying Cleanser,Murad,Skincare,Cleansers
2,467,P427415,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,Moisturizers
3,488,P428250,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,Treatments
4,673,P440651,Mini Acne Control Clarifying Cleanser,Murad,Skincare,Mini Size


In [ ]:
product_name_to_idx = dict(
    zip(
        content_products["product_name"],
        content_products["product_idx"]
    )
)

In [ ]:
len(product_name_to_idx)

8415

In [ ]:
idx_to_product = content_products.set_index(
    "product_idx"
)

In [ ]:
idx_to_product.head()

,product_id,product_name,brand_name,primary_category,price_usd
product_idx,,,,,
0,P473671,Fragrance Discovery Set,19-69,Fragrance,35.0
1,P473668,La Habana Eau de Parfum,19-69,Fragrance,195.0
2,P473662,Rainbow Bar Eau de Parfum,19-69,Fragrance,195.0
3,P473660,Kasbah Eau de Parfum,19-69,Fragrance,195.0
4,P473658,Purple Haze Eau de Parfum,19-69,Fragrance,195.0


In [ ]:
def recommend_similar(product_name, top_n=10):

    # get product index
    product_idx = product_name_to_idx[product_name]


    # get similarity scores
    similarity_scores = list(
        enumerate(content_similarity[product_idx])
    )


    # sort highest similarity first
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x:x[1],
        reverse=True
    )


    # remove itself
    similarity_scores = similarity_scores[1:top_n+1]


    # extract indexes
    product_indices = [
        i[0] for i in similarity_scores
    ]


    result = idx_to_product.loc[
        product_indices
    ].copy()


    result["similarity_score"] = [
        round(i[1],3)
        for i in similarity_scores
    ]


    return result[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "price_usd",
            "similarity_score"
        ]
    ]

In [ ]:
products.iloc[100][
    [
        "product_name"
    ]
]

product_name    GENIUS Ultimate Anti-Aging Vitamin C+ Serum
Name: 100, dtype: object

In [ ]:
recommend_similar(
    "GENIUS Ultimate Anti-Aging Vitamin C+ Serum",
    top_n=5
)

,product_id,product_name,brand_name,primary_category,price_usd,similarity_score
product_idx,,,,,,
108,P388262,GENIUS Ultimate Anti-Aging Eye Cream,Algenist,Skincare,74.0,0.487
90,P421277,GENIUS Liquid Collagen Serum,Algenist,Skincare,115.0,0.373
116,P456990,Mini GENIUS Liquid Collagen,Algenist,Skincare,25.0,0.369
97,P500716,10 Day Results Kit,Algenist,Skincare,88.0,0.366
94,P384537,GENIUS Ultimate Anti-Aging Cream,Algenist,Skincare,112.0,0.342


In [ ]:
joblib.dump(
    content_products,
    model_dir/"recommendation/content_based/product_mapping.pkl"
)

['c:\\Users\\ujjwa\\OneDrive\\Documents\\beauty-recommender\\models\\recommendation\\content_based\\product_mapping.pkl']

In [ ]:
item_user_matrix.shape

(2122, 104861)

In [ ]:
cf_products.head()

,product_idx,product_id,product_name,brand_name,primary_category,secondary_category
0,199,P394639,The True Cream Aqua Bomb,belif,Skincare,Moisturizers
1,236,P4016,Acne Control Clarifying Cleanser,Murad,Skincare,Cleansers
2,467,P427415,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,Moisturizers
3,488,P428250,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,Treatments
4,673,P440651,Mini Acne Control Clarifying Cleanser,Murad,Skincare,Mini Size


In [ ]:
cf_product_to_idx = dict(
    zip(
        cf_products["product_id"],
        cf_products["product_idx"]
    )
)

In [ ]:
len(cf_product_to_idx)

2122

In [ ]:
cf_idx_to_product = cf_products.merge(
    products[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "price_usd"
        ]
    ],
    on="product_id",
    how="left"
)

cf_idx_to_product.head()

,product_idx,product_id,product_name_x,brand_name_x,primary_category_x,secondary_category,product_name_y,brand_name_y,primary_category_y,price_usd
0,199,P394639,The True Cream Aqua Bomb,belif,Skincare,Moisturizers,The True Cream Aqua Bomb,belif,Skincare,38.0
1,236,P4016,Acne Control Clarifying Cleanser,Murad,Skincare,Cleansers,Acne Control Clarifying Cleanser,Murad,Skincare,36.0
2,467,P427415,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,Moisturizers,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,10.9
3,488,P428250,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,Treatments,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,19.0
4,673,P440651,Mini Acne Control Clarifying Cleanser,Murad,Skincare,Mini Size,Mini Acne Control Clarifying Cleanser,Murad,Skincare,14.0


In [ ]:
cf_idx_to_product.head()

,product_idx,product_id,product_name_x,brand_name_x,primary_category_x,secondary_category,product_name_y,brand_name_y,primary_category_y,price_usd
0,199,P394639,The True Cream Aqua Bomb,belif,Skincare,Moisturizers,The True Cream Aqua Bomb,belif,Skincare,38.0
1,236,P4016,Acne Control Clarifying Cleanser,Murad,Skincare,Cleansers,Acne Control Clarifying Cleanser,Murad,Skincare,36.0
2,467,P427415,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,Moisturizers,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,10.9
3,488,P428250,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,Treatments,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,19.0
4,673,P440651,Mini Acne Control Clarifying Cleanser,Murad,Skincare,Mini Size,Mini Acne Control Clarifying Cleanser,Murad,Skincare,14.0


In [ ]:
cf_idx_to_product = cf_idx_to_product.rename(
    columns={
        "product_name_y": "product_name",
        "brand_name_y": "brand_name",
        "primary_category_y": "primary_category"
    }
)

In [ ]:
cf_idx_to_product = cf_idx_to_product[
    [
        "product_idx",
        "product_id",
        "product_name",
        "brand_name",
        "primary_category",
        "price_usd"
    ]
]

In [ ]:
cf_idx_to_product.head()

,product_idx,product_id,product_name,brand_name,primary_category,price_usd
0,199,P394639,The True Cream Aqua Bomb,belif,Skincare,38.0
1,236,P4016,Acne Control Clarifying Cleanser,Murad,Skincare,36.0
2,467,P427415,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,10.9
3,488,P428250,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,19.0
4,673,P440651,Mini Acne Control Clarifying Cleanser,Murad,Skincare,14.0


In [ ]:
def recommend_collaborative(product_id, top_n=10):

    # convert product id to matrix index
    product_idx = cf_product_to_idx[product_id]


    # find nearest products
    distances, indices = item_knn.kneighbors(
        item_user_matrix[product_idx],
        n_neighbors=top_n+1
    )


    recommendations = []


    for idx, distance in zip(
        indices[0][1:],
        distances[0][1:]
    ):

        recommendations.append(idx)


    result = cf_idx_to_product[
        cf_idx_to_product["product_idx"]
        .isin(recommendations)
    ].copy()


    score_map = dict(
        zip(
            recommendations,
            1 - distances[0][1:]
        )
    )


    result["similarity_score"] = (
        result["product_idx"]
        .map(score_map)
    )


    return result[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "price_usd",
            "similarity_score"
        ]
    ].sort_values(
        "similarity_score",
        ascending=False
    )

In [ ]:
cf_products.head()

,product_idx,product_id,product_name,brand_name,primary_category,secondary_category
0,199,P394639,The True Cream Aqua Bomb,belif,Skincare,Moisturizers
1,236,P4016,Acne Control Clarifying Cleanser,Murad,Skincare,Cleansers
2,467,P427415,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,Moisturizers
3,488,P428250,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,Treatments
4,673,P440651,Mini Acne Control Clarifying Cleanser,Murad,Skincare,Mini Size


In [ ]:
product_id = "P394639"

In [ ]:
recommend_collaborative(
    product_id,
    top_n=5
)

,product_id,product_name,brand_name,primary_category,price_usd,similarity_score
180,P449175,Aqua Bomb Hydrating Toner,belif,Skincare,30.0,0.183974
173,P394624,The True Cream Moisturizing Bomb,belif,Skincare,38.0,0.119819
224,P475908,Aqua Bomb Overnight Lip Mask,belif,Skincare,22.0,0.103303
263,P462336,Aqua Bomb Cleansing Balm,belif,Skincare,38.0,0.093206
462,P472050,Cold Plunge Pore Remedy Moisturizer with BHA/LHA,OLEHENRIKSEN,Skincare,51.0,0.059706


In [ ]:
popularity_map = products[
    [
        "product_id",
        "popularity_score"
    ]
].copy()

popularity_map.head()

,product_id,popularity_score
0,P473671,6.590390
1,P473668,6.460104
2,P473662,6.410906
3,P473660,6.459573
4,P473658,5.971970


In [ ]:
hybrid_products = products[
    [
        "product_id",
        "product_name",
        "brand_name",
        "primary_category",
        "price_usd",
        "popularity_score"
    ]
].copy()


hybrid_products.head()

,product_id,product_name,brand_name,primary_category,price_usd,popularity_score
0,P473671,Fragrance Discovery Set,19-69,Fragrance,35.0,6.590390
1,P473668,La Habana Eau de Parfum,19-69,Fragrance,195.0,6.460104
2,P473662,Rainbow Bar Eau de Parfum,19-69,Fragrance,195.0,6.410906
3,P473660,Kasbah Eau de Parfum,19-69,Fragrance,195.0,6.459573
4,P473658,Purple Haze Eau de Parfum,19-69,Fragrance,195.0,5.971970


In [ ]:
hybrid_products.shape

(8494, 6)

In [ ]:
from sklearn.preprocessing import MinMaxScaler


scaler = MinMaxScaler()


hybrid_products["popularity_norm"] = (
    scaler.fit_transform(
        hybrid_products[
            ["popularity_score"]
        ]
    )
)


hybrid_products.head()

,product_id,product_name,brand_name,primary_category,price_usd,popularity_score,popularity_norm
0,P473671,Fragrance Discovery Set,19-69,Fragrance,35.0,6.590390,0.570271
1,P473668,La Habana Eau de Parfum,19-69,Fragrance,195.0,6.460104,0.556643
2,P473662,Rainbow Bar Eau de Parfum,19-69,Fragrance,195.0,6.410906,0.551497
3,P473660,Kasbah Eau de Parfum,19-69,Fragrance,195.0,6.459573,0.556587
4,P473658,Purple Haze Eau de Parfum,19-69,Fragrance,195.0,5.971970,0.505583


In [ ]:
from sklearn.preprocessing import MinMaxScaler



In [ ]:
popularity_products = hybrid_products[
    [
        "product_id",
        "product_name",
        "brand_name",
        "primary_category",
        "price_usd",
        "popularity_norm"
    ]
].copy()


popularity_products.head()

,product_id,product_name,brand_name,primary_category,price_usd,popularity_norm
0,P473671,Fragrance Discovery Set,19-69,Fragrance,35.0,0.570271
1,P473668,La Habana Eau de Parfum,19-69,Fragrance,195.0,0.556643
2,P473662,Rainbow Bar Eau de Parfum,19-69,Fragrance,195.0,0.551497
3,P473660,Kasbah Eau de Parfum,19-69,Fragrance,195.0,0.556587
4,P473658,Purple Haze Eau de Parfum,19-69,Fragrance,195.0,0.505583


In [ ]:
product_id_to_idx = dict(
    zip(
        content_products["product_id"],
        content_products["product_idx"]
    )
)


idx_to_product_id = dict(
    zip(
        content_products["product_idx"],
        content_products["product_id"]
    )
)

In [ ]:
product_id_to_idx["P394639"]

511

In [ ]:
content_product_to_idx = dict(
    zip(
        content_products["product_id"],
        content_products["product_idx"]
    )
)

In [ ]:
cf_product_to_idx = dict(
    zip(
        cf_products["product_id"],
        cf_products["product_idx"]
    )
)

In [ ]:
content_product_to_idx["P394639"]

511

In [ ]:
cf_product_to_idx["P394639"]

199

In [ ]:
def get_content_scores(product_id, top_n=20):

    idx = content_product_to_idx[product_id]

    similarity_scores = content_similarity[idx]

    similar_indices = np.argsort(
        similarity_scores
    )[::-1][1:top_n+1]


    result = pd.DataFrame({
        "product_idx": similar_indices,
        "content_score": similarity_scores[similar_indices]
    })


    result["product_id"] = (
        result["product_idx"]
        .map(idx_to_product_id)
    )


    return result[
        [
            "product_id",
            "content_score"
        ]
    ]

In [ ]:
get_content_scores(
    "P394639",
    top_n=5
)

,product_id,content_score
0,P394624,0.988049
1,P504852,0.765340
2,P504888,0.726518
3,P449175,0.609050
4,P435801,0.606271


In [ ]:
def get_collab_scores(product_id, top_n=20):

    cf_idx = cf_product_to_idx[product_id]


    distances, indices = item_knn.kneighbors(
        item_user_matrix.getrow(cf_idx),
        n_neighbors=top_n + 1
    )


    recommendations = indices[0][1:]
    scores = 1 - distances[0][1:]


    result = pd.DataFrame({
        "cf_idx": recommendations,
        "collab_score": scores
    })


    result["product_id"] = (
        result["cf_idx"]
        .map(
            dict(
                zip(
                    cf_products["product_idx"],
                    cf_products["product_id"]
                )
            )
        )
    )


    return result[
        [
            "product_id",
            "collab_score"
        ]
    ]

In [ ]:
get_collab_scores(
    "P394639",
    top_n=5
)

,product_id,collab_score
0,P449175,0.183974
1,P394624,0.119819
2,P475908,0.103303
3,P462336,0.093206
4,P472050,0.059706


In [ ]:
def hybrid_recommend(product_id, top_n=10):

    # =====================================================
    # 1. Get scores from each recommendation model
    # =====================================================

    content_scores = get_content_scores(
        product_id,
        top_n=50
    )


    collab_scores = get_collab_scores(
        product_id,
        top_n=50
    )


    # =====================================================
    # 2. Merge Content + Collaborative Scores
    # =====================================================

    hybrid = content_scores.merge(
        collab_scores,
        on="product_id",
        how="outer"
    )


    # Missing recommendations from a model get score 0

    hybrid["content_score"] = (
        hybrid["content_score"]
        .fillna(0)
    )


    hybrid["collab_score"] = (
        hybrid["collab_score"]
        .fillna(0)
    )


    # =====================================================
    # 3. Add Popularity Score
    # =====================================================

    hybrid = hybrid.merge(
        hybrid_products[
            [
                "product_id",
                "popularity_norm"
            ]
        ],
        on="product_id",
        how="left"
    )


    hybrid["popularity_norm"] = (
        hybrid["popularity_norm"]
        .fillna(0)
    )


    # =====================================================
    # 4. Calculate Final Hybrid Score
    # =====================================================

    hybrid["final_score"] = (
        0.5 * hybrid["content_score"]
        +
        0.3 * hybrid["collab_score"]
        +
        0.2 * hybrid["popularity_norm"]
    )


    # =====================================================
    # 5. Remove the Input Product
    # =====================================================

    hybrid = hybrid[
        hybrid["product_id"] != product_id
    ]


    # =====================================================
    # 6. Add Product Metadata
    # =====================================================

    hybrid = hybrid.merge(
        hybrid_products[
            [
                "product_id",
                "product_name",
                "brand_name",
                "primary_category",
                "price_usd"
            ]
        ],
        on="product_id",
        how="left"
    )


    # =====================================================
    # 7. Final Ranked Output
    # =====================================================

    return hybrid[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "price_usd",
            "final_score",
            "content_score",
            "collab_score",
            "popularity_norm"
        ]
    ].sort_values(
        "final_score",
        ascending=False
    ).head(top_n)

In [ ]:
hybrid_recommend(
    "P394639",
    top_n=10
)

,product_id,product_name,brand_name,primary_category,price_usd,final_score,content_score,collab_score,popularity_norm
9,P394624,The True Cream Moisturizing Bomb,belif,Skincare,38.0,0.702157,0.988049,0.119819,0.860933
34,P449175,Aqua Bomb Hydrating Toner,belif,Skincare,30.0,0.504493,0.609050,0.183974,0.723877
87,P504852,Aqua Bomb Everyday Hydration Set,belif,Skincare,38.0,0.479410,0.765340,0.000000,0.483698
68,P475908,Aqua Bomb Overnight Lip Mask,belif,Skincare,22.0,0.455287,0.569943,0.103303,0.696622
88,P504888,Aqua Bomb Protect & Glow Travel Kit,belif,Skincare,27.0,0.451482,0.726518,0.000000,0.441116
20,P422905,Moisturizing Eye Bomb with Squalane,belif,Skincare,48.0,0.447137,0.529084,0.056711,0.827909
28,P435801,Aqua Bomb Mist,belif,Skincare,38.0,0.431222,0.606271,0.000000,0.640431
30,P444057,Aqua Bomb Jelly Cleanser,belif,Skincare,30.0,0.429446,0.563097,0.029244,0.695625
26,P433443,Aqua Bomb Sleeping Mask,belif,Skincare,38.0,0.429370,0.526355,0.040510,0.770199
76,P481952,Aqua Bomb Hydrating Body Moisturizer with Niac...,belif,Bath & Body,30.0,0.411300,0.564436,0.000000,0.645412


In [ ]:
hybrid_dir = (
    model_dir
    / "recommendation"
    / "hybrid"
)

hybrid_dir.mkdir(
    parents=True,
    exist_ok=True
)

hybrid_dir

WindowsPath('c:/Users/ujjwa/OneDrive/Documents/beauty-recommender/models/recommendation/hybrid')

In [ ]:
joblib.dump(
    hybrid_products,
    hybrid_dir/"hybrid_products.pkl"
)

['c:\\Users\\ujjwa\\OneDrive\\Documents\\beauty-recommender\\models\\recommendation\\hybrid\\hybrid_products.pkl']

In [ ]:
hybrid_config = {
    "content_weight":0.5,
    "collab_weight":0.3,
    "popularity_weight":0.2
}

In [ ]:
joblib.dump(
    hybrid_config,
    hybrid_dir/"hybrid_config.pkl"
)

['c:\\Users\\ujjwa\\OneDrive\\Documents\\beauty-recommender\\models\\recommendation\\hybrid\\hybrid_config.pkl']

In [ ]:
loaded_products = joblib.load(
    hybrid_dir/"hybrid_products.pkl"
)

loaded_config = joblib.load(
    hybrid_dir/"hybrid_config.pkl"
)

In [ ]:
loaded_products.head()

,product_id,product_name,brand_name,primary_category,price_usd,popularity_score,popularity_norm
0,P473671,Fragrance Discovery Set,19-69,Fragrance,35.0,6.590390,0.570271
1,P473668,La Habana Eau de Parfum,19-69,Fragrance,195.0,6.460104,0.556643
2,P473662,Rainbow Bar Eau de Parfum,19-69,Fragrance,195.0,6.410906,0.551497
3,P473660,Kasbah Eau de Parfum,19-69,Fragrance,195.0,6.459573,0.556587
4,P473658,Purple Haze Eau de Parfum,19-69,Fragrance,195.0,5.971970,0.505583


In [13]:
reviews.columns

Index(['author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name', 'brand_name', 'price_usd',
       'positive_signal', 'engagement_score', 'review_length', 'recency_days',
       'time_weight', 'interaction_strength'],
      dtype='object')